# Chapitre 15 · Passer à l'échelle : la vraie chose

Notebook du chapitre 15 de *Construire un LLM de zéro* (Partie III « S'entraîner comme un labo »).

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et prêt à
exécuter. Lis, exécute, modifie pour voir. À la fin, la section **Exercices** : quatre
défis à trous, du plus simple au plus costaud, validés par des `assert`. Les corrigés
vivent dans le notebook solution (`solutions/partie_3_sentrainer_comme_un_labo/`).

Ce chapitre est le SEUL du livre où une partie du contenu (le nœud cloud) demande de
louer du GPU. **Mais ce notebook, lui, tourne ENTIÈREMENT sur CPU, sans GPU.** On utilise
le backend **gloo** de PyTorch pour faire communiquer 2 processus locaux : c'est
exactement le mécanisme d'une grappe de GPU (où le backend serait nccl), en petit et
gratuit. Tu vas VOIR les gradients se synchroniser, chiffres à l'appui.

## Setup

PyTorch et de quoi lancer des processus. Rien à installer de plus. L'utilitaire
`lance_torchrun` démarre un script en N workers (via `torchrun`) et récupère leur sortie.

In [ ]:
import os
import sys
import subprocess
import textwrap

import torch
import torch.distributed as dist

print("PyTorch", torch.__version__)
print("gloo disponible :", dist.is_gloo_available())
print("CUDA (GPU) disponible :", torch.cuda.is_available())

# Ce notebook tourne ENTIÈREMENT sur CPU, sans GPU. On utilise le backend gloo
# pour faire communiquer 2 processus locaux : c'est le même mécanisme que sur une
# grappe de GPU (où le backend serait nccl), en petit et gratuit.

# Petit utilitaire : lancer un script en N workers via torchrun, et renvoyer la sortie.
def lance_torchrun(script, nproc=2, port=29600, args=None):
    cmd = [sys.executable, "-m", "torch.distributed.run",
           f"--nproc_per_node={nproc}", "--nnodes=1", f"--master_port={port}", script]
    if args:
        cmd += args
    env = dict(os.environ, OMP_NUM_THREADS="1")
    r = subprocess.run(cmd, capture_output=True, text=True, env=env, timeout=300)
    # torchrun mélange stdout/stderr ; on garde les lignes utiles.
    lignes = (r.stdout + r.stderr).splitlines()
    return r.returncode, lignes

## 1. Le jour où le rêve devient une facture

Deux raisons distinctes d'aller au multi-GPU : des données trop grosses pour le temps
disponible (parallélisme de données, le sujet du chapitre) ou un modèle trop gros pour
la mémoire d'un GPU (parallélisme de modèle, nommé pas construit). La machine louée :
un pod (les GPU), un volume persistant (les checkpoints) et un compteur qui tourne.
Ici, rien à louer : tout ce qui suit tourne sur ton CPU.

## 2. DDP : plusieurs répliques qui restent d'accord

Le DDP en trois gestes : une copie identique du modèle par GPU (un **worker**), un
mini-lot différent par worker à chaque étape, et une mise en commun des gradients
(leur **moyenne**) avant chaque correction, pour que toutes les copies restent
identiques. Cette mise en commun s'appelle l'**all-reduce**.

### 2.2 L'all-reduce écrit à la main

Chaque worker voit des données différentes, calcule donc un gradient différent.
L'all-reduce additionne les gradients de tous les workers, puis on divise par leur
nombre : chacun repart avec la MÊME moyenne, donc reste une réplique identique.
On écrit le script du worker sur disque, puis on le lance en 2 processus.

In [ ]:
ALLREDUCE = textwrap.dedent("""
    import os
    import torch, torch.distributed as dist
    import torch.nn as nn, torch.nn.functional as F

    rank = int(os.environ["RANK"]); world = int(os.environ["WORLD_SIZE"])
    dist.init_process_group("gloo", rank=rank, world_size=world)

    torch.manual_seed(42)                 # MÊME init -> répliques identiques
    model = nn.Linear(4, 1, bias=False)
    torch.manual_seed(100 + rank)         # données PROPRES à chaque worker
    x, y = torch.randn(8, 4), torch.randn(8, 1)
    F.mse_loss(model(x), y).backward()
    g = model.weight.grad

    before = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), before, dst=0)
    dist.all_reduce(g, op=dist.ReduceOp.SUM)   # chaque worker reçoit la SOMME
    g /= world                                  # ... transformée en MOYENNE
    after = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), after, dst=0)

    if rank == 0:
        for r in range(world):
            print(f"worker {r} | AVANT all-reduce : grad = {before[r].item():+.4f}")
        moy = sum(b.item() for b in before) / world
        print(f"moyenne des gradients = {moy:+.4f}")
        for r in range(world):
            print(f"worker {r} | APRES all-reduce : grad = {after[r].item():+.4f}")
    dist.destroy_process_group()
""")
with open("allreduce_demo.py", "w") as f:
    f.write(ALLREDUCE)

code, lignes = lance_torchrun("allreduce_demo.py", nproc=2, port=29601)
print("returncode :", code)
for l in lignes:
    if "all-reduce" in l or "moyenne" in l:
        print(l)

**Ce que tu dois voir** : les deux workers ont des gradients différents (+2.0211 et
-0.6770), leur moyenne vaut +0.6721, et APRÈS l'all-reduce les deux portent cette même
moyenne. La grappe est soudée. Réflexe `shape` : l'all-reduce ne change pas la forme
du gradient, il en remplace les valeurs, place pour place.

### 2.3 DistributedDataParallel : le même travail, automatique

Dans la vraie vie, tu n'écris jamais l'all-reduce à la main. `DistributedDataParallel`
le déclenche tout seul pendant `backward()`. On vérifie qu'il donne EXACTEMENT le même
gradient que la main (+0.6721).

In [ ]:
DDP_AUTO = textwrap.dedent("""
    import os
    import torch, torch.distributed as dist
    import torch.nn as nn, torch.nn.functional as F
    from torch.nn.parallel import DistributedDataParallel as DDP

    rank = int(os.environ["RANK"]); world = int(os.environ["WORLD_SIZE"])
    dist.init_process_group("gloo", rank=rank, world_size=world)

    torch.manual_seed(42)
    model = nn.Linear(4, 1, bias=False)
    ddp_model = DDP(model)                 # aucune ligne d'all-reduce à écrire
    torch.manual_seed(100 + rank)
    x, y = torch.randn(8, 4), torch.randn(8, 1)
    F.mse_loss(ddp_model(x), y).backward() # DDP déclenche l'all-reduce ICI

    g = model.weight.grad
    vals = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), vals, dst=0)
    if rank == 0:
        for r in range(world):
            print(f"worker {r} | grad DDP = {vals[r].item():+.4f}")
    dist.destroy_process_group()
""")
with open("ddp_auto_demo.py", "w") as f:
    f.write(DDP_AUTO)

code, lignes = lance_torchrun("ddp_auto_demo.py", nproc=2, port=29602)
print("returncode :", code)
for l in lignes:
    if "grad DDP" in l:
        print(l)

**Ce que tu dois voir** : `grad DDP = +0.6721` sur les deux workers, exactement le
résultat de l'all-reduce à la main. DDP ne fait pas de magie : il automatise ce que
tu viens d'écrire.

### 2.4 Le cas qui échoue : sans synchro, la grappe se casse en silence

On fait tourner 5 pas d'entraînement, une fois AVEC la synchronisation des gradients,
une fois SANS, et on compare le poids final de chaque worker. Sans synchro, les workers
divergent : ce ne sont plus deux répliques, mais deux modèles différents. Et rien ne plante.

In [ ]:
DIVERGENCE = textwrap.dedent("""
    import os
    import torch, torch.distributed as dist
    import torch.nn as nn, torch.nn.functional as F

    def run(sync):
        torch.manual_seed(42)
        model = nn.Linear(4, 1, bias=False)
        opt = torch.optim.SGD(model.parameters(), lr=0.1)
        rank, world = dist.get_rank(), dist.get_world_size()
        torch.manual_seed(100 + rank)
        batches = [(torch.randn(8, 4), torch.randn(8, 1)) for _ in range(5)]
        for x, y in batches:
            F.mse_loss(model(x), y).backward()
            if sync:
                g = model.weight.grad
                dist.all_reduce(g, op=dist.ReduceOp.SUM); g /= world
            opt.step(); opt.zero_grad()
        return model.weight.detach()[0, 0].item()

    rank = int(os.environ["RANK"]); world = int(os.environ["WORLD_SIZE"])
    dist.init_process_group("gloo", rank=rank, world_size=world)
    w_sync = run(sync=True)
    w_nosync = run(sync=False)

    s = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    n = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(torch.tensor([w_sync]), s, dst=0)
    dist.gather(torch.tensor([w_nosync]), n, dst=0)
    if rank == 0:
        for r in range(world):
            print(f"worker {r} | AVEC synchro : poids final = {s[r].item():+.4f}")
        for r in range(world):
            print(f"worker {r} | SANS synchro : poids final = {n[r].item():+.4f}")
    dist.destroy_process_group()
""")
with open("divergence_demo.py", "w") as f:
    f.write(DIVERGENCE)

code, lignes = lance_torchrun("divergence_demo.py", nproc=2, port=29603)
print("returncode :", code)
for l in lignes:
    if "synchro" in l:
        print(l)

**Ce que tu dois voir** : AVEC synchro, les deux workers finissent sur le même poids
(+0.1172) ; SANS synchro, ils divergent (+0.2190 contre +0.0731). Rien ne plante : la
grappe continue de tourner, mais elle ne calcule plus un modèle cohérent. Le premier
réflexe de diagnostic sur une grappe : vérifier que les poids sont identiques entre workers.

### 2.5 Découper les données : le DistributedSampler

Reste à garantir que les workers ne voient pas deux fois le même exemple. Le
`DistributedSampler` découpe le dataset en `world_size` parts sans recouvrement et
donne à chaque worker sa part, et seulement la sienne. `sampler.set_epoch(epoch)`
change le mélange à chaque époque, de façon cohérente entre workers.

In [ ]:
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

dataset = TensorDataset(torch.arange(32))   # 32 exemples numérotés de 0 à 31
world_size = 2

parts = {}
for rank in range(world_size):
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=16, sampler=sampler)
    sampler.set_epoch(0)              # re-mélange, mais de façon cohérente entre workers
    parts[rank] = list(sampler)       # les indices que CE worker verra à l'époque 0
    print(f"worker {rank} | ses 16 indices (époque 0) : {parts[rank]}")

print("vus deux fois :", (set(parts[0]) & set(parts[1])) or "aucun")
print("tout le dataset couvert :", set(parts[0]) | set(parts[1]) == set(range(len(dataset))))

# set_epoch change le mélange d'une époque à l'autre :
sampler = DistributedSampler(dataset, num_replicas=world_size, rank=0, shuffle=True)
sampler.set_epoch(1)
print(f"worker 0 | ses 16 indices (époque 1) : {list(sampler)}")

## 3. On allume, on lance, on éteint

La discipline de coût d'un labo : prototyper sur Colab AVANT de louer, lancer dans
tmux, checkpointer souvent sur le volume persistant, rapatrier les checkpoints, puis
détruire le pod ET le volume. On ne débogue jamais sur une instance facturée à l'heure.

### 3.2 torchrun : lancer N workers d'une commande

Le fichier `train_ddp.py` (à côté de ce notebook) entraîne le petit GPT des fables en DDP.
Le MÊME fichier tourne sur ton CPU (backend gloo, 2 workers) et sur un nœud cloud
(backend nccl, N GPU) : il détecte cuda tout seul. On le lance ici en subprocess, sur CPU,
pour prouver qu'il tourne, sans louer un seul GPU.

Sur un nœud cloud, tu lancerais à la place :
```bash
torchrun --nproc_per_node=8 --nnodes=1 train_ddp.py --steps 2000
```

In [ ]:
# Le script train_ddp.py vit dans le dossier partie_3 du dépôt (à côté du notebook exercice).
# On le cherche depuis le dossier courant en remontant l'arborescence, puis on le lance
# en subprocess avec 2 workers CPU : la PREUVE qu'il est prêt pour le cloud.
def trouver_train_ddp():
    base = os.getcwd()
    for _ in range(6):  # remonter jusqu'à 6 niveaux
        for sous in ("", "partie_3_sentrainer_comme_un_labo",
                     os.path.join("construire-un-llm-de-zero", "partie_3_sentrainer_comme_un_labo")):
            cand = os.path.join(base, sous, "train_ddp.py")
            if os.path.exists(cand):
                return cand
        base = os.path.dirname(base)
    return None

chemin = trouver_train_ddp()
print("train_ddp.py trouvé :", chemin)
assert chemin is not None, "train_ddp.py introuvable (il doit être dans partie_3_sentrainer_comme_un_labo/)"

code, lignes = lance_torchrun(chemin, nproc=2, port=29604, args=["--steps", "200"])
print("returncode :", code)
for l in lignes:
    if any(k in l for k in ["backend", "step", "termine", "checkpoint"]):
        print(l)

**Ce que tu dois voir** : `backend=gloo world_size=2`, une loss qui descend
(2.127 -> 0.668), un checkpoint écrit par le seul rang 0. Change `--nproc_per_node` et
mets des GPU : c'est le même script qui entraîne pour de vrai sur le nœud loué.

## 4. Toucher les lois d'échelle du doigt

La perte suit une loi de puissance du calcul investi (Kaplan, 2020) : les labos
entraînent des petits modèles, tracent la droite et extrapolent le grand. Chinchilla
(2022) ajoute l'équilibre : environ **20 tokens d'entraînement par paramètre** pour un
budget donné. Et les labos actuels vont « plus petit, plus longtemps » (LLaMA), parce
qu'un modèle plus petit coûte moins cher à servir.

In [ ]:
# La règle Chinchilla : environ 20 tokens d'entraînement par paramètre.
for n_params in (125e6, 1e9, 70e9):
    n_tokens = 20 * n_params
    print(f"{n_params/1e9:7.3f} Md de paramètres -> ~{n_tokens/1e9:8.1f} Md de tokens (compute-optimal)")

## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●). Chaque
cellule marquée `# TODO(toi)` contient un trou ; complète-le, puis exécute la cellule
de validation (`assert`) qui suit : si elle passe sans erreur, c'est gagné.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta place.
Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi. Les réponses
sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · La moyenne des gradients — niveau ●

L'all-reduce, réduit à son arithmétique : chaque worker envoie son gradient, tous
reçoivent la somme, qu'on divise par le nombre de workers. Écris cette opération en
pur PyTorch, sans processus ni gloo : une fonction qui reçoit la liste des gradients
locaux et renvoie ce que chaque worker porte APRÈS l'all-reduce.

In [ ]:
# TODO(toi) : écris allreduce_moyenne(grads) qui renvoie la liste des gradients
# APRÈS all-reduce : chaque worker porte la MOYENNE de tous les gradients.
# (grads est une liste de tenseurs, un par worker ; renvoie une liste de même longueur.)
def allreduce_moyenne(grads):
    ...

In [ ]:
# Validation : all-reduce = somme / nombre de workers.
apres = allreduce_moyenne([torch.tensor([2.0211]), torch.tensor([-0.6770])])
assert len(apres) == 2, "chaque worker doit repartir avec un gradient"
assert all(torch.allclose(a, torch.tensor([0.67205])) for a in apres), "chacun doit porter la moyenne"
apres3 = allreduce_moyenne([torch.tensor([3.0]), torch.tensor([0.0]), torch.tensor([0.0])])
assert all(torch.allclose(a, torch.tensor([1.0])) for a in apres3), "ça doit marcher pour 3 workers aussi"
print(f"All-reduce en petit OK : +2.0211 et -0.6770 -> {apres[0].item():+.5f} partout")

### Exercice 2 · L'all-reduce à la main, en vrai — niveau ●●

Maintenant avec 2 vrais processus qui communiquent par gloo. Le script ci-dessous est
celui de la section 2.2, MOINS les deux lignes qui soudent la grappe. Remplace les
`# TODO(toi)` par l'all-reduce à la main (additionner les gradients de tous les
workers, puis diviser par leur nombre), supprime le `raise`, puis lance la validation.
Essaie de mémoire avant de remonter à la section 2.2.

In [ ]:
EXO_ALLREDUCE = textwrap.dedent("""
    import os
    import torch, torch.distributed as dist
    import torch.nn as nn, torch.nn.functional as F

    rank = int(os.environ["RANK"]); world = int(os.environ["WORLD_SIZE"])
    dist.init_process_group("gloo", rank=rank, world_size=world)

    torch.manual_seed(42)                 # MÊME init -> répliques identiques
    model = nn.Linear(4, 1, bias=False)
    torch.manual_seed(100 + rank)         # données PROPRES à chaque worker
    x, y = torch.randn(8, 4), torch.randn(8, 1)
    F.mse_loss(model(x), y).backward()
    g = model.weight.grad

    before = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), before, dst=0)
    # TODO(toi) : additionner les gradients de tous les workers (SUM)
    # TODO(toi) : diviser par le nombre de workers pour obtenir la MOYENNE
    raise NotImplementedError('Écris l all-reduce à la main, puis supprime cette ligne')
    after = [torch.zeros(1) for _ in range(world)] if rank == 0 else None
    dist.gather(g[0, 0].reshape(1), after, dst=0)

    if rank == 0:
        for r in range(world):
            print(f"worker {r} | AVANT all-reduce : grad = {before[r].item():+.4f}")
        for r in range(world):
            print(f"worker {r} | APRES all-reduce : grad = {after[r].item():+.4f}")
    dist.destroy_process_group()
""")
with open("exo_allreduce.py", "w") as f:
    f.write(EXO_ALLREDUCE)
print("exo_allreduce.py écrit : complète les TODO ci-dessus, puis valide.")

In [ ]:
# Validation : l'all-reduce à la main, exécuté par 2 vrais processus.
code_exo, lignes_exo = lance_torchrun("exo_allreduce.py", nproc=2, port=29611)
assert code_exo == 0, "le script a échoué : as-tu remplacé les TODO (et supprimé le raise) ?"
avant = [float(l.rsplit("=", 1)[1]) for l in lignes_exo if "AVANT all-reduce" in l]
apres = [float(l.rsplit("=", 1)[1]) for l in lignes_exo if "APRES all-reduce" in l]
assert len(avant) == 2 and len(apres) == 2, "il manque des lignes de sortie"
assert abs(apres[0] - apres[1]) < 1e-6, "les deux workers doivent porter la MÊME valeur"
assert abs(apres[0] - sum(avant) / 2) < 1e-3, "la valeur portée doit être la MOYENNE des gradients"
print(f"All-reduce OK : {avant[0]:+.4f} et {avant[1]:+.4f} -> {apres[0]:+.4f} sur les deux workers")

### Exercice 3 · Chacun sa part : le DistributedSampler — niveau ●●

Une grappe de 2 workers, un dataset de 32 exemples. Construis la part de chaque worker
avec `DistributedSampler` (mélange activé), et vérifie la promesse de la section 2.5 :
deux parts de 16, sans recouvrement, qui couvrent tout le dataset.

In [ ]:
from torch.utils.data import TensorDataset, DistributedSampler

dataset_exo = TensorDataset(torch.arange(32))

# TODO(toi) : construis part_0 et part_1, les listes d'indices vus par le worker 0
# et le worker 1 d'une grappe de 2 (num_replicas=2, rank=..., shuffle=True).
part_0 = ...
part_1 = ...

In [ ]:
# Validation : découpage sans recouvrement.
assert sorted(map(len, [part_0, part_1])) == [16, 16], "deux parts de 16 exemples chacune"
assert set(part_0) & set(part_1) == set(), "aucun exemple ne doit être vu par les deux workers"
assert set(part_0) | set(part_1) == set(range(32)), "aucun exemple ne doit être oublié"
print("DistributedSampler OK : 32 exemples, 2 parts de 16, sans recouvrement")

### Exercice 4 · Le point Chinchilla-optimal — niveau ●●●

Combine deux règles que tu connais : le coût d'entraînement C ≈ 6 · N · D FLOPs
(chapitre 14) et l'équilibre Chinchilla D = 20 · N (20 tokens par paramètre). Pour un
budget C fixé, résous les deux équations : quelle taille de modèle N, et combien de
tokens D ? (Il te faut une racine carrée : `math.sqrt`.)

In [ ]:
import math

# TODO(toi) : écris chinchilla_optimal(budget_flops) qui renvoie (n_params, n_tokens),
# le couple compute-optimal obtenu en combinant C = 6 * N * D et D = 20 * N.
def chinchilla_optimal(budget_flops):
    ...

In [ ]:
# Validation : le point Chinchilla-optimal.
n, d = chinchilla_optimal(1.2e20)
assert abs(n - 1e9) / 1e9 < 1e-6, f"pour 1.2e20 FLOPs, N attendu ~1e9, obtenu {n:.3e}"
assert abs(d - 2e10) / 2e10 < 1e-6, f"pour 1.2e20 FLOPs, D attendu ~2e10, obtenu {d:.3e}"
n2, d2 = chinchilla_optimal(6 * 70e9 * 1.4e12)   # le budget de Chinchilla lui-même
assert abs(d2 / n2 - 20) < 1e-6, "le ratio tokens/paramètres doit valoir 20"
assert abs(n2 - 70e9) / 70e9 < 1e-3, f"N attendu ~70 Md, obtenu {n2:.3e}"
print(f"Chinchilla OK : 1.2e20 FLOPs -> N = {n:.2e} paramètres, D = {d:.2e} tokens")

## Verdict

Quatre validations vertes : tu sais ce qui soude une grappe. Passer à l'échelle, ce
n'est pas une autre physique : c'est la même, **répliquée**. Un modèle par GPU, des
gradients qui se moyennent par all-reduce après chaque étape, et la discipline de
n'allumer que ce qui sert (on allume, on lance, on éteint). Tu viens de tout voir
marcher **sur ton CPU**, chiffres à l'appui, sans louer un GPU.

Reste à savoir si le modèle est **bon** : la loss d'entraînement peut mentir. C'est le
chapitre 16, « Évaluer sérieusement ».